In [1]:
%pip -q install duckdb pyarrow aiohttp

from google.colab import drive, userdata
from pathlib import Path
import asyncio
import json
import os
import random
import time

import aiohttp
import duckdb
import pandas as pd

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PARQUET_PATH = Path(
    "/content/drive/MyDrive/Language Detection/"
    "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
)

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing from Colab Secrets")

OPENROUTER_CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_GENERATION_URL = "https://openrouter.ai/api/v1/generation"

MODELS = {
    "gpt_oss_120b": "openai/gpt-oss-120b",
    "hermes_4_70b": "nousresearch/hermes-4-70b",
    "llama_3_3_70b": "meta-llama/llama-3.3-70b-instruct",
}

LANGUAGE_NAMES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

ALLOWED_CODES = tuple(LANGUAGE_NAMES)
SEGMENTS_PER_LANGUAGE = 25
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 4

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

con = duckdb.connect()
con.execute("SET threads TO 4")
con.execute("SET preserve_insertion_order = false")

def load_language_segments(language_code):
    if language_code not in LANGUAGE_NAMES:
        raise ValueError(language_code)

    frame = con.execute(
        """
        WITH sessions AS (
            SELECT
                gamesession_id,
                url,
                TRY_CAST(created_at AS TIMESTAMP) AS created_at,
                transcript_segments
            FROM read_parquet(?)
            WHERE
                lang_detected = ?
                AND transcript_segments IS NOT NULL
                AND len(transcript_segments) > 0
        ),
        exploded AS (
            SELECT
                gamesession_id,
                url,
                created_at,
                generate_subscripts(transcript_segments, 1) AS segment_index,
                UNNEST(transcript_segments) AS segment
            FROM sessions
        ),
        eligible AS (
            SELECT
                gamesession_id,
                url,
                created_at,
                segment_index,
                TRIM(segment.text) AS segment_text,
                len(
                    regexp_extract_all(
                        TRIM(segment.text),
                        '[\\p{L}\\p{N}]+'
                    )
                ) AS word_count
            FROM exploded
            WHERE
                segment.text IS NOT NULL
                AND TRIM(segment.text) <> ''
        ),
        ranked_sessions AS (
            SELECT
                gamesession_id,
                MAX(created_at) AS created_at,
                COUNT(*) AS eligible_segments
            FROM eligible
            WHERE word_count >= 4
            GROUP BY gamesession_id
            HAVING COUNT(*) >= ?
            ORDER BY created_at DESC NULLS LAST, gamesession_id DESC
            LIMIT 1
        )
        SELECT
            ? AS dataset_language,
            e.gamesession_id,
            e.url,
            e.segment_index,
            e.segment_text
        FROM eligible e
        INNER JOIN ranked_sessions s USING (gamesession_id)
        WHERE e.word_count >= 4
        ORDER BY e.segment_index
        LIMIT ?
        """,
        [
            PARQUET_PATH.as_posix(),
            language_code,
            SEGMENTS_PER_LANGUAGE,
            language_code,
            SEGMENTS_PER_LANGUAGE,
        ],
    ).df()

    if len(frame) != SEGMENTS_PER_LANGUAGE:
        raise RuntimeError(
            f"{language_code}: expected {SEGMENTS_PER_LANGUAGE} segments, found {len(frame)}"
        )

    frame.insert(0, "benchmark_row_id", range(len(frame)))
    return frame

def normalize_prediction(value):
    if not isinstance(value, str):
        return None

    value = value.strip().lower()
    aliases = {
        "english": "en",
        "german": "de",
        "deutsch": "de",
        "french": "fr",
        "français": "fr",
        "francais": "fr",
        "portuguese": "pt",
        "português": "pt",
        "portugues": "pt",
        "spanish": "es",
        "español": "es",
        "espanol": "es",
        "russian": "ru",
        "русский": "ru",
    }

    if value in ALLOWED_CODES:
        return value

    if value in aliases:
        return aliases[value]

    cleaned = value.replace("`", "").replace('"', "").replace("'", "").strip()

    if cleaned in ALLOWED_CODES:
        return cleaned

    if cleaned in aliases:
        return aliases[cleaned]

    for code in ALLOWED_CODES:
        if cleaned.startswith(code + " ") or cleaned.startswith(code + "\n"):
            return code

    return None

def build_payload(model_id, text):
    payload = {
        "model": model_id,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Detect the language of the transcript. "
                    "Return exactly one ISO 639-1 code from: "
                    "en, de, fr, pt, es, ru. No explanation."
                ),
            },
            {
                "role": "user",
                "content": text,
            },
        ],
        "temperature": 0,
    }

    if model_id == "openai/gpt-oss-120b":
        payload["reasoning"] = {"effort": "low"}

    return payload

async def post_chat(session, model_name, model_id, row):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        started = time.perf_counter()

        try:
            async with session.post(
                OPENROUTER_CHAT_URL,
                headers=HEADERS,
                json=build_payload(model_id, row.segment_text),
            ) as response:
                elapsed = time.perf_counter() - started
                data = await response.json(content_type=None)

                if response.status == 200:
                    message = ((data.get("choices") or [{}])[0].get("message") or {})
                    content = message.get("content")
                    usage = data.get("usage") or {}

                    return {
                        "benchmark_row_id": row.benchmark_row_id,
                        "model_name": model_name,
                        "model_id": model_id,
                        "prediction": normalize_prediction(content),
                        "raw_prediction": content,
                        "generation_id": data.get("id"),
                        "http_status": response.status,
                        "request_duration_seconds": elapsed,
                        "prompt_tokens": usage.get("prompt_tokens"),
                        "completion_tokens": usage.get("completion_tokens"),
                        "total_tokens": usage.get("total_tokens"),
                        "response_cost_usd": usage.get("cost"),
                        "error": None,
                    }

                error = data.get("error")
                last_error = error.get("message") if isinstance(error, dict) else str(error or data)

                if response.status not in {408, 409, 429, 500, 502, 503, 504}:
                    break

                retry_after = response.headers.get("Retry-After")
                delay = float(retry_after) if retry_after else min(30.0, 2 ** attempt + random.random())
                await asyncio.sleep(delay)

        except Exception as exc:
            elapsed = time.perf_counter() - started
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRIES:
                await asyncio.sleep(min(30.0, 2 ** attempt + random.random()))

    return {
        "benchmark_row_id": row.benchmark_row_id,
        "model_name": model_name,
        "model_id": model_id,
        "prediction": None,
        "raw_prediction": None,
        "generation_id": None,
        "http_status": None,
        "request_duration_seconds": elapsed,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "response_cost_usd": None,
        "error": last_error,
    }

async def benchmark_language(language_code):
    segments = load_language_segments(language_code)
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
    records = []

    async with aiohttp.ClientSession(timeout=timeout) as session:
        for row in segments.itertuples(index=False):
            for model_name, model_id in MODELS.items():
                record = await post_chat(session, model_name, model_id, row)
                records.append(record)

    raw = pd.DataFrame(records)

    predictions = (
        raw.pivot(
            index="benchmark_row_id",
            columns="model_name",
            values="prediction",
        )
        .rename(columns=lambda name: f"openrouter_{name}_language")
        .reset_index()
    )

    result = segments.merge(predictions, on="benchmark_row_id", how="left")
    return result, raw

async def fetch_generation(session, generation_id):
    started = time.perf_counter()

    try:
        async with session.get(
            OPENROUTER_GENERATION_URL,
            headers=HEADERS,
            params={"id": generation_id},
        ) as response:
            lookup_duration = time.perf_counter() - started
            payload = await response.json(content_type=None)

            if response.status != 200:
                return {
                    "generation_id": generation_id,
                    "generation_lookup_http_status": response.status,
                    "generation_lookup_seconds": lookup_duration,
                    "generation_error": str(payload),
                }

            data = payload.get("data") or payload
            return {
                "generation_id": generation_id,
                "generation_lookup_http_status": response.status,
                "generation_lookup_seconds": lookup_duration,
                "generation_model": data.get("model"),
                "generation_provider": data.get("provider_name") or data.get("provider"),
                "generation_prompt_tokens": data.get("tokens_prompt"),
                "generation_completion_tokens": data.get("tokens_completion"),
                "generation_total_cost_usd": data.get("total_cost"),
                "generation_latency_ms": data.get("latency"),
                "generation_generation_time_ms": data.get("generation_time"),
                "generation_error": None,
            }

    except Exception as exc:
        return {
            "generation_id": generation_id,
            "generation_lookup_http_status": None,
            "generation_lookup_seconds": time.perf_counter() - started,
            "generation_error": f"{type(exc).__name__}: {exc}",
        }

async def build_accounting(raw_results):
    source = raw_results[
        raw_results["generation_id"].notna()
    ].copy()

    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS)
    metadata = []

    async with aiohttp.ClientSession(timeout=timeout) as session:
        for generation_id in source["generation_id"].drop_duplicates():
            metadata.append(await fetch_generation(session, generation_id))

    metadata = pd.DataFrame(metadata)

    if metadata.empty:
        detailed = source.copy()
    else:
        detailed = source.merge(metadata, on="generation_id", how="left")

    summary = (
        detailed.groupby(["model_name", "model_id"], as_index=False)
        .agg(
            api_calls=("generation_id", "size"),
            successful_predictions=("prediction", lambda x: x.notna().sum()),
            prompt_tokens=("prompt_tokens", "sum"),
            completion_tokens=("completion_tokens", "sum"),
            total_tokens=("total_tokens", "sum"),
            response_cost_usd=("response_cost_usd", "sum"),
            request_duration_seconds=("request_duration_seconds", "sum"),
            avg_request_duration_seconds=("request_duration_seconds", "mean"),
            official_generation_cost_usd=("generation_total_cost_usd", "sum"),
            avg_official_latency_ms=("generation_latency_ms", "mean"),
            avg_official_generation_time_ms=("generation_generation_time_ms", "mean"),
        )
    )

    return detailed, summary

print("Ready")
print("Parquet:", PARQUET_PATH)
print("Models :", ", ".join(MODELS))
print("Flow   : run language cells one by one, then accounting cells one by one")


Mounted at /content/drive
Ready
Parquet: /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Models : gpt_oss_120b, hermes_4_70b, llama_3_3_70b
Flow   : run language cells one by one, then accounting cells one by one


In [2]:
english_benchmark, english_raw = await benchmark_language("en")

print("English")
print("Segments:", len(english_benchmark))
print("API calls:", len(english_raw))
print("Successful predictions:", int(english_raw["prediction"].notna().sum()))
print("Failed predictions:", int(english_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(english_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

English
Segments: 25
API calls: 75
Successful predictions: 75
Failed predictions: 0


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language
0,0,en,141268501,https://www.twitch.tv/videos/2854311727,2,"Don't be talking, shut my door.",en,en,en
1,1,en,141268501,https://www.twitch.tv/videos/2854311727,3,We'll start it.,en,en,en
2,2,en,141268501,https://www.twitch.tv/videos/2854311727,4,We'll start with the basics.,en,en,en
3,3,en,141268501,https://www.twitch.tv/videos/2854311727,5,actually showed up. You guys owe me ten bucks.,en,en,en
4,4,en,141268501,https://www.twitch.tv/videos/2854311727,6,"personal. It's just, uh, we're stuck in Groundhog Day out here.",en,en,en
5,5,en,141268501,https://www.twitch.tv/videos/2854311727,7,"trying to, you know, not go crazy..",en,en,en
6,6,en,141268501,https://www.twitch.tv/videos/2854311727,8,"Okay, it's simple really, shout next and the guys will let the survivor in.",en,en,en
7,7,en,141268501,https://www.twitch.tv/videos/2854311727,9,"Next survivor. First, do a simple inspection.",en,en,en
8,8,en,141268501,https://www.twitch.tv/videos/2854311727,10,Take the flashlight from the table.,en,en,en
9,9,en,141268501,https://www.twitch.tv/videos/2854311727,11,Take your time. Match what you see against the symptom chart.,en,en,en


In [3]:
german_benchmark, german_raw = await benchmark_language("de")

print("German")
print("Segments:", len(german_benchmark))
print("API calls:", len(german_raw))
print("Successful predictions:", int(german_raw["prediction"].notna().sum()))
print("Failed predictions:", int(german_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(german_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

German
Segments: 25
API calls: 75
Successful predictions: 75
Failed predictions: 0


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language
0,0,de,141192575,https://www.twitch.tv/videos/2852368077,1,"Der kann mich gar nicht erwischt, nie im Leben.",de,de,de
1,1,de,141192575,https://www.twitch.tv/videos/2852368077,2,ich hab's gerade gesehen.,de,de,de
2,2,de,141192575,https://www.twitch.tv/videos/2852368077,3,"Eigentlich hat das Gebäude, als wir offiziell in der Zwiebel waren.",de,de,de
3,3,de,141192575,https://www.twitch.tv/videos/2852368077,4,"Ich seh' halt nicht, keine Ahnung, wo der ist.",de,de,de
4,4,de,141192575,https://www.twitch.tv/videos/2852368077,5,"Irgendwo Richtung Turm drin, irgendwo da hinten, so ein bisschen weiter links.",de,de,de
5,5,de,141192575,https://www.twitch.tv/videos/2852368077,6,"einen haben wir am Tachel, einer ist noch unten drin ja ja da snipet er die Lachen",de,de,de
6,6,de,141192575,https://www.twitch.tv/videos/2852368077,7,"traurige ja gib mir ein Bredi ja ein richtiger Lack jetzt habe ich gepisst das, ich hätte dich geholt",de,de,de
7,7,de,141192575,https://www.twitch.tv/videos/2852368077,8,"ich weiß nicht, hier haben die die selber gibt's das das war das, was du deinen eigenen Strömen liebst warte mal ich schau mal Das ist jetzt Beste.",de,de,de
8,8,de,141192575,https://www.twitch.tv/videos/2852368077,10,"Die hat ja noch einer, jetzt können sie Tank und Psych nochmal.",de,de,de
9,9,de,141192575,https://www.twitch.tv/videos/2852368077,11,"Oh nein, ja er snipet immer noch wo da oben.",de,de,de


In [4]:
french_benchmark, french_raw = await benchmark_language("fr")

print("French")
print("Segments:", len(french_benchmark))
print("API calls:", len(french_raw))
print("Successful predictions:", int(french_raw["prediction"].notna().sum()))
print("Failed predictions:", int(french_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(french_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

French
Segments: 25
API calls: 75
Successful predictions: 75
Failed predictions: 0


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language
0,0,fr,141267561,https://www.twitch.tv/videos/2854116559,2,What you guys doing ?,en,en,en
1,1,fr,141267561,https://www.twitch.tv/videos/2854116559,6,What you guys doing ?,en,en,en
2,2,fr,141267561,https://www.twitch.tv/videos/2854116559,7,"Allez mec Attendez Yo Damien, t le premier ?",fr,fr,fr
3,3,fr,141267561,https://www.twitch.tv/videos/2854116559,8,"Ouais sur tiktak t le premier, même sur Twitch je pense En tout cas le premier message Euh, y'en perd pas de temps Aujourd'hui on essaie d'avancer putain de capa de merde Comme ça c'est fait, il nous manquera quoi ?",fr,fr,fr
4,4,fr,141267561,https://www.twitch.tv/videos/2854116559,9,"Là, il'on y s'extraire avec l'autre un, ça nous donnera la peluche, et après il manquera juste l'achantique, et problème de l'achantique, c que je sais pas comment je vais le pour la trouver, mais on va tenter, on va tenter.",fr,fr,fr
5,5,fr,141267561,https://www.twitch.tv/videos/2854116559,10,"Yo Pilouf,'as va ou quoi ? Comment va le Pilouf ?",fr,fr,fr
6,6,fr,141267561,https://www.twitch.tv/videos/2854116559,11,"Allez, ça part comme ça les gars. Un second. Ça raconte quoi Pilouf ?",fr,fr,fr
7,7,fr,141267561,https://www.twitch.tv/videos/2854116559,12,"J'espère que tu vas bien Damien. Ah d'ailleurs, je voulais te faire un truc.",fr,fr,fr
8,8,fr,141267561,https://www.twitch.tv/videos/2854116559,13,"Ah bah tranquille mon Pilouf. J'peux pas t'entendre, j'suis au taf, moi un pouce avec un grand sourire pour me dire bonjour La petit pouce Putain, travailler un dimanche, terrible",fr,fr,fr
9,9,fr,141267561,https://www.twitch.tv/videos/2854116559,14,"Mais putain, Pilouf alors Pilouf tu reprends lundi, enfin demain les cours",fr,fr,fr


In [5]:
portuguese_benchmark, portuguese_raw = await benchmark_language("pt")

print("Portuguese")
print("Segments:", len(portuguese_benchmark))
print("API calls:", len(portuguese_raw))
print("Successful predictions:", int(portuguese_raw["prediction"].notna().sum()))
print("Failed predictions:", int(portuguese_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(portuguese_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Portuguese
Segments: 25
API calls: 75
Successful predictions: 75
Failed predictions: 0


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language
0,0,pt,141268410,https://www.twitch.tv/videos/2854267348,1,FUERTA DE RAHIM Ta,es,es,es
1,1,pt,141268410,https://www.twitch.tv/videos/2854267348,2,de pé E Fui FUERTA RAHIM E ASSISTA FUERTA FAZ FUERTA RAHIM FUERTA RAHIM,pt,pt,pt
2,2,pt,141268410,https://www.twitch.tv/videos/2854267348,3,FUERTA RAHIM porta aberta e isso,pt,pt,pt
3,3,pt,141268410,https://www.twitch.tv/videos/2854267348,4,ok então estou bebendo um café hoje seria o dia do chill stream mas essa semana eu não joguei a LoL eu queria jogar a LoL eu queria jogar a LoL o chill stream significa que eu decido então se eu quero jogar LoL eu a LoL e vocês pegam no cu aquele eu mais alto talvez,pt,pt,pt
4,4,pt,141268410,https://www.twitch.tv/videos/2854267348,5,Como você Como você Galaxi?,pt,pt,pt
5,5,pt,141268410,https://www.twitch.tv/videos/2854267348,6,"Você fazendo tudo, você está certo. Como você Pasta?",pt,pt,pt
6,6,pt,141268410,https://www.twitch.tv/videos/2854267348,7,"Como você está? Shadow? Tudo bem, garotos? Razy, como você Tchau garotos, tchau a você bem?",pt,pt,pt
7,7,pt,141268410,https://www.twitch.tv/videos/2854267348,8,"O Urco desde esta manhã, oh Deus, desta manhã...",es,pt,pt
8,8,pt,141268410,https://www.twitch.tv/videos/2854267348,9,"Eu não estava. E saiu também no meu schedule, não sei se você viu.",pt,pt,pt
9,9,pt,141268410,https://www.twitch.tv/videos/2854267348,10,"Que é feito, não mais a Dorari. Mas é a...",pt,pt,pt


In [6]:
spanish_benchmark, spanish_raw = await benchmark_language("es")

print("Spanish")
print("Segments:", len(spanish_benchmark))
print("API calls:", len(spanish_raw))
print("Successful predictions:", int(spanish_raw["prediction"].notna().sum()))
print("Failed predictions:", int(spanish_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(spanish_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Spanish
Segments: 25
API calls: 75
Successful predictions: 75
Failed predictions: 0


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language
0,0,es,141268448,https://www.twitch.tv/videos/2853628924,1,hola hola que tal gente silla estamos aquí otra vez un día más vamos al charlando,es,es,es
1,1,es,141268448,https://www.twitch.tv/videos/2853628924,2,hello como estáis como estáis ahora estoy bastante cansada pero tengo una cosa aquí estamos aquí cumplimos vale pues hoy vamos a jugar a los dinosaurios que me apetece un ratito aunque sea y ya luego vamos al balo en un ratito hoy no me quedaré mucho porque me apetece estar más en la gama todo esto me hecho una siesta cuando he llegado,es,es,es
2,2,es,141268448,https://www.twitch.tv/videos/2853628924,3,de tres horas y claro una mañana que hay que trabajar entonces prefiero irme tempranito que sabemos como acaba de irse tempranito pero bueno lo cuento igual.,es,es,es
3,3,es,141268448,https://www.twitch.tv/videos/2853628924,4,eh... Vale. ¿Dónde está? ¿Dónde está?,es,es,es
4,4,es,141268448,https://www.twitch.tv/videos/2853628924,6,"Let's go. Vamos allá, continúen.",es,es,es
5,5,es,141268448,https://www.twitch.tv/videos/2853628924,7,"Vale, voy limpiar un momentito rápido el teclado.",es,es,es
6,6,es,141268448,https://www.twitch.tv/videos/2853628924,9,"Pero no lo que tenía que hacer, tipo, las dos canastas y los ladrillos de piedra vale,",es,es,es
7,7,es,141268448,https://www.twitch.tv/videos/2853628924,10,"ehm, los de piedra y las dos canastas, tengo que hacer nooo, una planta por la puta,",es,es,es
8,8,es,141268448,https://www.twitch.tv/videos/2853628924,11,"falta una planta ah ya tengo las dos vale, ok, eh vamos a reunirnos con Bryce",es,es,es
9,9,es,141268448,https://www.twitch.tv/videos/2853628924,12,"¿Por qué hay emisiones? ¿Qué más tenían? Aumentar la población, pero eso lo vas diciendo poco a",es,es,es


In [7]:
russian_benchmark, russian_raw = await benchmark_language("ru")

print("Russian")
print("Segments:", len(russian_benchmark))
print("API calls:", len(russian_raw))
print("Successful predictions:", int(russian_raw["prediction"].notna().sum()))
print("Failed predictions:", int(russian_raw["prediction"].isna().sum()))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(russian_benchmark)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Russian
Segments: 25
API calls: 75
Successful predictions: 75
Failed predictions: 0


,benchmark_row_id,dataset_language,gamesession_id,url,segment_index,segment_text,openrouter_gpt_oss_120b_language,openrouter_hermes_4_70b_language,openrouter_llama_3_3_70b_language
0,0,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,2,"ДИНАМИЧНАЯ Ooh-ooh, kiss my shit, kiss my",ru,ru,ru
1,1,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,3,"shit I heard you were talking shit And you didn't think that I would hear it People hear you talking, like",en,en,en
2,2,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,6,"Так я не понимаю, нахуй он сначала девочек за бодашкой врубил?",ru,ru,ru
3,3,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,7,"Чад, всем дарова, всем... Ты что, реально врубил, придурок? Блять, ты ебла нахуй, нахуй ты меня трахаешь, ебанашка Ростян, он реально врубил, Ростян?",ru,ru,ru
4,4,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,8,"Да, я врубил Да Короче, дарова, Чад, привет, всем Ну вот, так вам скажу, нас сегодня распаковка анимешных фигурок, анимешных фигурок из магазина popo-man или popo-ma popo-me-me-me вот",ru,ru,ru
5,5,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,9,"а, оффот? пишут, пишут, оффот, не надо стримить всё, тогда оффо, нахуй, чад, спасибо, всем пока короче, взяли конструктивного компотика, ну, такого рационального правильного, вкусного собрались в небольшой компании рациональных людей это окей Юра окей вот Юра вот Леша вот",ru,ru,ru
6,6,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,10,Рациональный Леша нас сегодня Леша нас сегодня диджей Леша диджей и Растян с танком Растян с танком я тебя могу вот что будет так что то попьем пивка распакуем игрушки и ляжем спать вот все не знаю кто спать сегодня денечек такой не сбылся,ru,ru,ru
7,7,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,11,спирит проебали блять моя ставка проебала блять сколько 30 6 тысяч 136 тысяч,ru,ru,ru
8,8,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,12,"Да, Санька, Санька, займись, Саша, займись.",ru,ru,ru
9,9,ru,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,13,"Вот, потихоньку кое-что, пивасик сейчас поглушим и распакуем.",ru,ru,ru


In [8]:
english_accounting, english_accounting_summary = await build_accounting(english_raw)

print("English accounting")
display(english_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        english_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


English accounting


,model_name,model_id,api_calls,successful_predictions,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,avg_request_duration_seconds,official_generation_cost_usd,avg_official_latency_ms,avg_official_generation_time_ms
0,gpt_oss_120b,openai/gpt-oss-120b,25,25,2851,861,3712,0.000443,28.721710,1.148868,0.000443,713.72,1378.16
1,hermes_4_70b,nousresearch/hermes-4-70b,25,25,1525,50,1575,0.000218,7.567468,0.302699,0.000218,86.64,90.72
2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,25,25,1608,50,1658,0.000309,34.270741,1.370830,0.000309,474.48,563.64


,benchmark_row_id,model_name,model_id,prediction,generation_id,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,generation_provider,generation_prompt_tokens,generation_completion_tokens,generation_total_cost_usd,generation_latency_ms,generation_generation_time_ms,generation_error
0,0,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787715906-3UQDdIb0GsGcdmlVMw7n,112,47,159,0.000022,1.164887,DigitalOcean,39,40,0.000022,501,1971,None
1,0,hermes_4_70b,nousresearch/hermes-4-70b,en,gen-1787715908-KAopK5LePcuYUjpDNnPZ,57,2,59,0.000008,0.436420,Nebius,39,1,0.000008,85,88,None
2,0,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,en,gen-1787715909-4WZQrzUa2zJbFq2Xtaa9,57,2,59,0.000006,0.377182,DeepInfra,82,1,0.000006,268,296,None
3,1,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787715909-GSPiwGbglJwm6b8TcarY,111,22,133,0.000015,1.235745,SiliconFlow,35,19,0.000015,1163,3127,None
4,1,hermes_4_70b,nousresearch/hermes-4-70b,en,gen-1787715913-ABBzfTzDkOcA7jV4sThw,53,2,55,0.000008,0.330974,Nebius,35,1,0.000008,78,79,None
5,1,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,en,gen-1787715913-lmrbGOQfOJh1F2tJUYMH,53,2,55,0.000006,5.619453,DeepInfra,78,1,0.000006,363,399,None
6,2,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787715919-ZKNa39jDMxXvCOe4HE3N,113,30,143,0.000008,1.060940,AkashML,38,21,0.000008,536,1016,None
7,2,hermes_4_70b,nousresearch/hermes-4-70b,en,gen-1787715921-MMESpiHwTIDpI97MrSYy,55,2,57,0.000008,0.289083,Nebius,38,1,0.000008,85,86,None
8,2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,en,gen-1787715923-bAxvblmotrB5XRHIp9qu,55,2,57,0.000006,2.563894,DeepInfra,81,1,0.000006,519,555,None
9,3,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787715924-igPWWniXmf85wy1odAQQ,118,34,152,0.000021,1.199970,SiliconFlow,42,35,0.000021,1047,2343,None


In [9]:
german_accounting, german_accounting_summary = await build_accounting(german_raw)

print("German accounting")
display(german_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        german_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


German accounting


,model_name,model_id,api_calls,successful_predictions,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,avg_request_duration_seconds,official_generation_cost_usd,avg_official_latency_ms,avg_official_generation_time_ms
0,gpt_oss_120b,openai/gpt-oss-120b,25,25,2972,819,3791,0.000487,19.207319,0.768293,0.000487,598.44,1339.16
1,hermes_4_70b,nousresearch/hermes-4-70b,25,25,1716,50,1766,0.000243,7.539199,0.301568,0.000243,97.88,105.04
2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,25,25,1961,50,2011,0.000456,14.492114,0.579685,0.000456,572.08,639.44


,benchmark_row_id,model_name,model_id,prediction,generation_id,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,generation_provider,generation_prompt_tokens,generation_completion_tokens,generation_total_cost_usd,generation_latency_ms,generation_generation_time_ms,generation_error
0,0,gpt_oss_120b,openai/gpt-oss-120b,de,gen-1787716042-PAfll9sLyc2SuoIq8fD1,119,25,144,0.000040,0.949155,SambaNova,43,17,0.000040,740,746,None
1,0,hermes_4_70b,nousresearch/hermes-4-70b,de,gen-1787716043-ggN7DAXP7ueOB68PjN68,61,2,63,0.000009,0.275257,Nebius,43,1,0.000009,85,88,None
2,0,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,de,gen-1787716044-lr1bqrYdnMRSJi6OLHAn,81,2,83,0.000016,0.988406,AkashML,86,1,0.000016,461,768,None
3,1,gpt_oss_120b,openai/gpt-oss-120b,de,gen-1787716045-qZUcnWScZ4rikuFsScXr,112,39,151,0.000015,2.113512,Novita,37,29,0.000015,446,752,None
4,1,hermes_4_70b,nousresearch/hermes-4-70b,de,gen-1787716047-h7WvbcgXo9srsh3GPMDz,55,2,57,0.000008,0.259877,Nebius,37,1,0.000008,77,84,None
5,1,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,de,gen-1787716048-ibOLMuK2gsPJgvt34GXK,56,2,58,0.000016,0.410431,Crusoe,80,1,0.000016,258,354,None
6,2,gpt_oss_120b,openai/gpt-oss-120b,de,gen-1787716048-yD2W3vfQu2lvbIRCcAXr,123,49,172,0.000028,1.576066,SiliconFlow,48,49,0.000028,1157,3853,None
7,2,hermes_4_70b,nousresearch/hermes-4-70b,de,gen-1787716053-rAF7V8ohQoCr1eSZ7h1w,69,2,71,0.000010,0.272042,Nebius,48,1,0.000010,79,85,None
8,2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,de,gen-1787716053-BxvgAoSmcBh63zZwMtUn,69,2,71,0.000008,0.337635,DeepInfra,91,1,0.000008,341,376,None
9,3,gpt_oss_120b,openai/gpt-oss-120b,de,gen-1787716054-PmVDeSd3G3nQLgkKtZ7f,117,54,171,0.000025,0.665131,DigitalOcean,42,41,0.000025,536,1729,None


In [10]:
french_accounting, french_accounting_summary = await build_accounting(french_raw)

print("French accounting")
display(french_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        french_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


French accounting


,model_name,model_id,api_calls,successful_predictions,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,avg_request_duration_seconds,official_generation_cost_usd,avg_official_latency_ms,avg_official_generation_time_ms
0,gpt_oss_120b,openai/gpt-oss-120b,25,25,3239,777,4016,0.000415,28.326166,1.133047,0.000415,757.00,1358.48
1,hermes_4_70b,nousresearch/hermes-4-70b,25,25,1946,50,1996,0.000273,9.139899,0.365596,0.000273,147.40,152.52
2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,25,25,1975,48,2023,0.000359,34.557636,1.382305,0.000359,761.92,804.76


,benchmark_row_id,model_name,model_id,prediction,generation_id,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,generation_provider,generation_prompt_tokens,generation_completion_tokens,generation_total_cost_usd,generation_latency_ms,generation_generation_time_ms,generation_error
0,0,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787716140-pZAUFP8xovY6W17zARsp,109,34,143,0.000017,0.843087,DigitalOcean,36,24,0.000017,668,1479,None
1,0,hermes_4_70b,nousresearch/hermes-4-70b,en,gen-1787716142-vrh3FCN8y5XOn1PAokkg,53,2,55,0.000008,0.626319,Nebius,36,1,0.000008,420,425,None
2,0,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,en,gen-1787716143-M9BIB76DctYXpaKjqyBU,54,2,56,0.000013,5.918475,Parasail,79,1,0.000013,459,459,None
3,1,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787716149-RQKzMVGcI6mv84b8bbGE,109,34,143,0.000017,0.804269,DigitalOcean,36,21,0.000017,665,1612,None
4,1,hermes_4_70b,nousresearch/hermes-4-70b,en,gen-1787716151-3ODYDIlGKyWB5JZbCgQP,53,2,55,0.000008,0.704097,Nebius,36,1,0.000008,169,171,None
5,1,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,en,gen-1787716151-BS3UOXd33rGgfcZabvFY,53,2,55,0.000006,5.610390,DeepInfra,79,1,0.000006,244,268,None
6,2,gpt_oss_120b,openai/gpt-oss-120b,fr,gen-1787716157-LA6OPhkTp7FtD54YiwXk,106,40,146,0.000011,0.869716,DeepInfra,42,30,0.000011,521,1766,None
7,2,hermes_4_70b,nousresearch/hermes-4-70b,fr,gen-1787716159-26bFnDZpf5g0mvl2vhee,60,2,62,0.000009,0.302204,Nebius,42,1,0.000009,78,78,None
8,2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,fr,gen-1787716160-xEHMyIgEMlNbnrCIeu0R,61,2,63,0.000013,3.012991,Parasail,85,1,0.000013,420,420,None
9,3,gpt_oss_120b,openai/gpt-oss-120b,fr,gen-1787716163-gTnvNIHnui4d1vC0KpKW,163,37,200,0.000017,0.779961,Novita,85,26,0.000017,586,767,None


In [11]:
portuguese_accounting, portuguese_accounting_summary = await build_accounting(portuguese_raw)

print("Portuguese accounting")
display(portuguese_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        portuguese_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


Portuguese accounting


,model_name,model_id,api_calls,successful_predictions,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,avg_request_duration_seconds,official_generation_cost_usd,avg_official_latency_ms,avg_official_generation_time_ms
0,gpt_oss_120b,openai/gpt-oss-120b,25,25,3083,700,3783,0.000403,17.689171,0.707567,0.000403,514.96,1144.16
1,hermes_4_70b,nousresearch/hermes-4-70b,25,25,1729,50,1779,0.000245,7.521223,0.300849,0.000245,98.08,100.08
2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,25,25,1877,50,1927,0.000352,16.536202,0.661448,0.000352,572.40,593.52


,benchmark_row_id,model_name,model_id,prediction,generation_id,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,generation_provider,generation_prompt_tokens,generation_completion_tokens,generation_total_cost_usd,generation_latency_ms,generation_generation_time_ms,generation_error
0,0,gpt_oss_120b,openai/gpt-oss-120b,es,gen-1787716258-JHcZAv5ctOnjxc4X6wOz,101,41,142,0.000011,0.462326,DeepInfra,35,28,0.000011,432,1339,None
1,0,hermes_4_70b,nousresearch/hermes-4-70b,es,gen-1787716259-CX1EE3SwWu37mhZ5Bscz,56,2,58,0.000008,0.283467,Nebius,35,1,0.000008,82,82,None
2,0,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,es,gen-1787716260-YEVgCCJLdk0DFIsh3hGc,57,2,59,0.000008,1.853985,Novita,78,1,0.000008,1349,1349,None
3,1,gpt_oss_120b,openai/gpt-oss-120b,pt,gen-1787716262-JHxlzrhF7ttHeD70GtEQ,121,44,165,0.000012,0.511076,DeepInfra,49,34,0.000012,556,1817,None
4,1,hermes_4_70b,nousresearch/hermes-4-70b,pt,gen-1787716264-ag8D0vH1BH7UKkhRrTBF,79,2,81,0.000011,0.508863,Nebius,49,1,0.000011,314,314,None
5,1,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,pt,gen-1787716264-yGoXrEWMjbzqAU2i18IO,99,2,101,0.000034,0.276920,Cloudflare,92,1,0.000034,188,188,None
6,2,gpt_oss_120b,openai/gpt-oss-120b,pt,gen-1787716265-2Wa3esZaCn3XnESDMPEz,116,36,152,0.000010,1.396646,AkashML,39,27,0.000010,1127,1685,None
7,2,hermes_4_70b,nousresearch/hermes-4-70b,pt,gen-1787716267-yANmOZ4BLfkjuLNMOggV,59,2,61,0.000008,0.304254,Nebius,39,1,0.000008,88,90,None
8,2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,pt,gen-1787716267-V1tqfIVYQNgxYIcRreuz,59,2,61,0.000007,0.362785,DeepInfra,82,1,0.000007,268,319,None
9,3,gpt_oss_120b,openai/gpt-oss-120b,pt,gen-1787716267-GXhhuJKjjaa46W5XSVAa,173,18,191,0.000008,0.728556,AkashML,99,7,0.000008,529,708,None


In [12]:
spanish_accounting, spanish_accounting_summary = await build_accounting(spanish_raw)

print("Spanish accounting")
display(spanish_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        spanish_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


Spanish accounting


,model_name,model_id,api_calls,successful_predictions,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,avg_request_duration_seconds,official_generation_cost_usd,avg_official_latency_ms,avg_official_generation_time_ms
0,gpt_oss_120b,openai/gpt-oss-120b,25,25,3091,825,3916,0.000457,29.211812,1.168472,0.000457,912.20,1594.96
1,hermes_4_70b,nousresearch/hermes-4-70b,25,25,1780,50,1830,0.000251,19.625432,0.785017,0.000251,658.16,684.96
2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,25,25,1850,50,1900,0.000356,13.899291,0.555972,0.000356,457.24,493.60


,benchmark_row_id,model_name,model_id,prediction,generation_id,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,generation_provider,generation_prompt_tokens,generation_completion_tokens,generation_total_cost_usd,generation_latency_ms,generation_generation_time_ms,generation_error
0,0,gpt_oss_120b,openai/gpt-oss-120b,es,gen-1787716344-e2FeZkSd1fIPvn1cZwgg,123,59,182,0.000021,2.604990,Novita,52,52,0.000021,2408,3189,None
1,0,hermes_4_70b,nousresearch/hermes-4-70b,es,gen-1787716347-j4LVDmLmkHaiekUvo8jB,68,2,70,0.000010,10.014108,Nebius,52,1,0.000010,11884,12274,None
2,0,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,es,gen-1787716360-s6QrgMKQSBHikWYFxUbU,68,2,70,0.000007,0.355911,DeepInfra,95,1,0.000007,388,465,None
3,1,gpt_oss_120b,openai/gpt-oss-120b,es,gen-1787716360-YIVi5aXYPHCaRA13ECFu,185,13,198,0.000008,1.654626,AkashML,117,3,0.000008,1480,1570,None
4,1,hermes_4_70b,nousresearch/hermes-4-70b,es,gen-1787716362-V1KMQa9LcH1igYr6nCST,129,2,131,0.000018,0.295121,Nebius,117,1,0.000018,102,132,None
5,1,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,es,gen-1787716362-mXId9Q3LkODJ3ogubmeE,130,2,132,0.000018,0.683618,Novita,160,1,0.000018,505,505,None
6,2,gpt_oss_120b,openai/gpt-oss-120b,es,gen-1787716363-mBHCiJGYzJN7pZZwButF,142,25,167,0.000009,0.478681,CoreWeave,71,17,0.000009,515,903,None
7,2,hermes_4_70b,nousresearch/hermes-4-70b,es,gen-1787716364-sMZnp5qp19YJaoN1hX4R,88,2,90,0.000012,0.306572,Nebius,71,1,0.000012,106,111,None
8,2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,es,gen-1787716365-dScVnNhx2RjtJbLUQNcO,88,2,90,0.000009,0.352142,DeepInfra,114,1,0.000009,482,674,None
9,3,gpt_oss_120b,openai/gpt-oss-120b,es,gen-1787716365-wZIvIChf8eUR2nj9UPIl,102,44,146,0.000025,0.546750,Google,42,31,0.000025,347,463,None


In [13]:
russian_accounting, russian_accounting_summary = await build_accounting(russian_raw)

print("Russian accounting")
display(russian_accounting_summary)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        russian_accounting[
            [
                "benchmark_row_id",
                "model_name",
                "model_id",
                "prediction",
                "generation_id",
                "prompt_tokens",
                "completion_tokens",
                "total_tokens",
                "response_cost_usd",
                "request_duration_seconds",
                "generation_provider",
                "generation_prompt_tokens",
                "generation_completion_tokens",
                "generation_total_cost_usd",
                "generation_latency_ms",
                "generation_generation_time_ms",
                "generation_error",
            ]
        ]
    )


Russian accounting


,model_name,model_id,api_calls,successful_predictions,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,avg_request_duration_seconds,official_generation_cost_usd,avg_official_latency_ms,avg_official_generation_time_ms
0,gpt_oss_120b,openai/gpt-oss-120b,25,25,3312,850,4162,0.000511,19.920296,0.796812,0.000511,717.04,1643.88
1,hermes_4_70b,nousresearch/hermes-4-70b,25,25,2158,50,2208,0.000301,7.554907,0.302196,0.000301,112.24,117.52
2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,25,25,2250,50,2300,0.000348,15.520276,0.620811,0.000348,596.88,634.56


,benchmark_row_id,model_name,model_id,prediction,generation_id,prompt_tokens,completion_tokens,total_tokens,response_cost_usd,request_duration_seconds,generation_provider,generation_prompt_tokens,generation_completion_tokens,generation_total_cost_usd,generation_latency_ms,generation_generation_time_ms,generation_error
0,0,gpt_oss_120b,openai/gpt-oss-120b,ru,gen-1787716461-jFLkglb3WPGIDfHKhnKG,112,32,144,0.000036,0.365763,DeepInfra,44,25,0.000036,309,579,None
1,0,hermes_4_70b,nousresearch/hermes-4-70b,ru,gen-1787716462-8ApVIymHu7CTEP0l9TsE,67,2,69,0.000010,0.481010,Nebius,44,1,0.000010,295,301,None
2,0,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,ru,gen-1787716463-DygF6UzV3hPMYiYiSJA2,68,2,70,0.000010,0.699448,Novita,87,1,0.000010,523,523,None
3,1,gpt_oss_120b,openai/gpt-oss-120b,en,gen-1787716463-8wnqx4DuvMtUnN7NIWuz,116,26,142,0.000009,0.523361,DeepInfra,57,17,0.000009,700,1489,None
4,1,hermes_4_70b,nousresearch/hermes-4-70b,en,gen-1787716465-lcHLbeXRQjFp46WfSmyC,71,2,73,0.000010,0.287063,Nebius,57,1,0.000010,99,104,None
5,1,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,en,gen-1787716465-8sO9gp1l2uCYnsrmOd8q,91,2,93,0.000014,1.118761,AkashML,100,1,0.000014,935,953,None
6,2,gpt_oss_120b,openai/gpt-oss-120b,ru,gen-1787716466-GUugEqUyP9M5M27Y4AtJ,113,30,143,0.000009,0.420216,DeepInfra,59,18,0.000009,596,1587,None
7,2,hermes_4_70b,nousresearch/hermes-4-70b,ru,gen-1787716468-Xh7uKaK4Tw98uEdKXDGK,71,2,73,0.000010,0.282322,Nebius,59,1,0.000010,91,93,None
8,2,llama_3_3_70b,meta-llama/llama-3.3-70b-instruct,ru,gen-1787716469-agoVlDQaElA84EnfHnGl,91,2,93,0.000018,0.960527,AkashML,102,1,0.000018,791,793,None
9,3,gpt_oss_120b,openai/gpt-oss-120b,ru,gen-1787716470-xsJUrLdeAGUoKGkS81Mf,149,29,178,0.000010,0.505623,DeepInfra,97,17,0.000010,1008,1501,None
